# Laboratorium 4 (4 pkt.)

Celem czwartego laboratorium jest zapoznanie się oraz zaimplementowanie algorytmów głębokiego uczenia aktywnego. Zaimplementowane algorytmy będą testowane z wykorzystaniem wcześniej przygotowanych środowisk: *FrozenLake* i *Pacman* oraz środowiska z OpenAI - *CartPole*.


Dołączenie standardowych bibliotek

In [1]:
from collections import deque
import gym
import numpy as np
import random

if not hasattr(np, "bool8"):
    np.bool8 = np.bool_

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


Dołączenie bibliotek ze środowiskami:

In [2]:
from env.FrozenLakeMDP import frozenLake
from env.FrozenLakeMDPExtended import frozenLakeExtended


Dołączenie bibliotek do obsługi sieci neuronowych

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim

class DQNModel(nn.Module):
    def __init__(self, input_dim, output_dim, learning_rate, hidden_units=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_units),
            nn.ReLU(),
            nn.Linear(hidden_units, hidden_units),
            nn.ReLU(),
            nn.Linear(hidden_units, output_dim),
        )
        self.loss_fn = nn.MSELoss()
        self.optimizer = optim.AdamW(self.parameters(), lr=learning_rate)

    def forward(self, x):
        return self.net(x)

    def predict(self, state):
        self.eval()
        with torch.no_grad():
            state_tensor = torch.as_tensor(state, dtype=torch.float32)
            q_values = self(state_tensor).cpu().numpy()
        return q_values

    def fit(self, states, targets):
        self.train()
        states_tensor = torch.as_tensor(states, dtype=torch.float32)
        targets_tensor = torch.as_tensor(targets, dtype=torch.float32)
        
        self.optimizer.zero_grad()
        predictions = self(states_tensor)
        loss = self.loss_fn(predictions, targets_tensor)
        loss.backward()
        self.optimizer.step()
        return loss.item()

## Zadanie 1 - Deep Q-Network

<p style='text-align: justify;'>
Celem ćwiczenie jest zaimplementowanie algorytmu Deep Q-Network. Wartoscią oczekiwaną sieci jest:
\begin{equation}
        Q(s_t, a_t) = r_{t+1} + \gamma \text{max}_a Q(s_{t + 1}, a)
\end{equation}
</p>

In [4]:
class DQNAgent:
    def __init__(self, action_size, learning_rate, model):
        self.action_size = action_size
        self.memory = deque(maxlen=10000)
        self.gamma = 0.95    # discount rate
        self.epsilon = 1.0  # exploration rate
        self.epsilon_min = 0.01
        self.epsilon_decay = 0.95
        self.learning_rate = learning_rate
        self.model = model

    def remember(self, state, action, reward, next_state, done):
        #Function adds information to the memory about last action and its results
        self.memory.append((state, action, reward, next_state, done)) 

    def get_action(self, state):
        """
        Compute the action to take in the current state, including exploration.
        With probability self.epsilon, we should take a random action.
            otherwise - the best policy action (self.get_best_action).

        Note: To pick randomly from a list, use random.choice(list).
              To pick True or False with a given probablity, generate uniform number in [0, 1]
              and compare it with your probability
        """

        #
        # INSERT CODE HERE to get action in a given state (according to epsilon greedy algorithm)
        if random.uniform(0, 1) < self.epsilon:
            chosen_action = random.choice(range(self.action_size))
        else:
            chosen_action = self.get_best_action(state)
        #        
        
        return chosen_action

  
    def get_best_action(self, state):
        """
        Compute the best action to take in a state.
        """

        #
        # INSERT CODE HERE to get best possible action in a given state (remember to break ties randomly)
        #
        q_values = self.model.predict(state)
        best_value = np.max(q_values)
        best_indices = np.flatnonzero(q_values == best_value)
        best_action = int(np.random.choice(best_indices))

        return best_action

    def replay(self, batch_size):
        """
        Function learn network using randomly selected actions from the memory. 
        First calculates Q value for the next state and choose action with the biggest value.
        Target value is calculated according to:
                Q(s,a) := (r + gamma * max_a(Q(s', a)))
        except the situation when the next action is the last action, in such case Q(s, a) := r.
        In order to change only those weights responsible for chosing given action, the rest values should be those
        returned by the network for state state.
        The network should be trained on batch_size samples.
        """
        #
        # INSERT CODE HERE to train network
        #
        if len(self.memory) < batch_size:
            return

        # sample = state, action, reward, next_state, done
        mini_batch = random.sample(self.memory, batch_size)
        states = np.vstack([sample[0] for sample in mini_batch])
        next_states = np.vstack([sample[3] for sample in mini_batch])
        
        targets = self.model.predict(states)
        next_q = self.model.predict(next_states)

        for i, (_, action, reward, _, done) in enumerate(mini_batch):
            if done:
                target_value = reward
            else:
                target_value = reward + self.gamma * np.max(next_q[i])
            targets[i][action] = target_value

        self.model.fit(states, targets)

    def update_epsilon_value(self):
        #Every each epoch epsilon value should be updated according to equation: 
        #self.epsilon *= self.epsilon_decay, but the updated value shouldn't be lower then epsilon_min value
        self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)

Czas przygotować model sieci, która będzie się uczyła poruszania po środowisku *FrozenLake*, warstwa wejściowa powinna mieć tyle neuronów ile jest możlliwych stanów, warstwa wyjściowa tyle neuronów ile jest możliwych akcji do wykonania:

In [5]:
env = frozenLake("8x8")

state_size = env.get_number_of_states()
action_size = len(env.get_possible_actions(None))
learning_rate = 0.001

model = DQNModel(state_size, action_size, learning_rate)

 Czas nauczyć agenta poruszania się po środowisku *FrozenLake*, jako stan przyjmij wektor o liczbie elementów równej liczbie możliwych stanów, z wartością 1 ustawioną w komórce o indeksie równym aktualnemu stanowi, pozostałe elementy mają być wypełnione zerami:
* 1 pkt < 35 epok,
* 0.5 pkt < 60 epok,
* 0.25 pkt - w pozostałych przypadkach.

In [6]:
agent = DQNAgent(action_size, learning_rate, model)

agent.epsilon = 0.75
agent.epsilon_decay = 0.9

done = False
batch_size = 128
EPISODES = 10000
counter = 0
for e in range(EPISODES):

    summary = []
    for _ in range(100):
        total_reward = 0
        env_state = env.reset()
    
        #
        # INSERT CODE HERE to prepare appropriate format of the state for network
        #
        state = np.zeros(state_size, dtype=np.float32)
        state[env_state] = 1.0
        state = np.reshape(state, (1, state_size))
        
        for time in range(1000):
            action = agent.get_action(state)
            next_state_env, reward, done, _ = env.step(action)
            total_reward += reward

            #
            # INSERT CODE HERE to prepare appropriate format of the next state for network
            #
            next_state = np.zeros(state_size, dtype=np.float32)
            next_state[next_state_env] = 1.0
            next_state = np.reshape(next_state, (1, state_size))

            #add to experience memory
            agent.remember(state, action, reward, next_state, done)
            state = next_state
            if done:
                break

        #
        # INSERT CODE HERE to train network if in the memory is more samples then size of the batch
        #
        if len(agent.memory) > batch_size:
            agent.replay(batch_size)
        
        summary.append(total_reward)
        
    agent.update_epsilon_value()
    print("epoch #{}\tmean reward = {:.3f}\tepsilon = {:.3f}".format(e, np.mean(summary), agent.epsilon))
    if np.mean(summary) > 0.9:
        print ("You Win!")
        break

epoch #0	mean reward = 0.000	epsilon = 0.675
epoch #1	mean reward = 0.010	epsilon = 0.608
epoch #2	mean reward = 0.060	epsilon = 0.547
epoch #3	mean reward = 0.130	epsilon = 0.492
epoch #4	mean reward = 0.210	epsilon = 0.443
epoch #5	mean reward = 0.320	epsilon = 0.399
epoch #6	mean reward = 0.210	epsilon = 0.359
epoch #7	mean reward = 0.370	epsilon = 0.323
epoch #8	mean reward = 0.280	epsilon = 0.291
epoch #9	mean reward = 0.570	epsilon = 0.262
epoch #10	mean reward = 0.790	epsilon = 0.235
epoch #11	mean reward = 0.730	epsilon = 0.212
epoch #12	mean reward = 0.740	epsilon = 0.191
epoch #13	mean reward = 0.590	epsilon = 0.172
epoch #14	mean reward = 0.410	epsilon = 0.154
epoch #15	mean reward = 0.780	epsilon = 0.139
epoch #16	mean reward = 0.900	epsilon = 0.125
epoch #17	mean reward = 0.870	epsilon = 0.113
epoch #18	mean reward = 0.810	epsilon = 0.101
epoch #19	mean reward = 0.870	epsilon = 0.091
epoch #20	mean reward = 0.860	epsilon = 0.082
epoch #21	mean reward = 0.810	epsilon = 0.07

Czas przygotować model sieci, która będzie się uczyła poruszania po środowisku *FrozenLakeExtended*, tym razem stan nie jest określany poprzez pojedynczą liczbę, a przez 3 tablice:
* pierwsza zawierająca informacje o celu,
* druga zawierająca informacje o dziurach,
* trzecia zawierająca informację o położeniu gracza.

In [7]:
env = frozenLakeExtended("4x4")

sample_state = env.reset()
if isinstance(sample_state, (list, tuple)) and len(sample_state) == 3:
    state_size = int(np.concatenate([np.array(part).flatten() for part in sample_state]).size)
else:
    state_size = int(np.array(sample_state).flatten().size)

action_size = len(env.get_possible_actions(None))
learning_rate = 0.001

model = DQNModel(state_size, action_size, learning_rate)

 Czas nauczyć agenta poruszania się po środowisku *FrozenLakeExtended*, jako stan przyjmij wektor składający się ze wszystkich trzech tablic (2 pkt.):

In [8]:
agent = DQNAgent(action_size, learning_rate, model)

agent.epsilon = 0.75
agent.epsilon_decay = 0.9

done = False
batch_size = 128
EPISODES = 2000
counter = 0
for e in range(EPISODES):
    summary = []
    for _ in range(100):
        total_reward = 0
        env_state = env.reset()
    
        #
        # INSERT CODE HERE to prepare appropriate format of the state for network
        #
        if isinstance(env_state, (list, tuple)) and len(env_state) == 3:
            state = np.concatenate([np.array(part).flatten() for part in env_state]).astype(np.float32)
        else:
            state = np.array(env_state, dtype=np.float32).flatten()
        state = np.reshape(state, (1, -1))
        
        for time in range(1000):
            action = agent.get_action(state)
            next_state_env, reward, done, _ = env.step(action)
            total_reward += reward

            #
            # INSERT CODE HERE to prepare appropriate format of the next state for network
            #
            if isinstance(next_state_env, (list, tuple)) and len(next_state_env) == 3:
                next_state = np.concatenate([np.array(part).flatten() for part in next_state_env]).astype(np.float32)
            else:
                next_state = np.array(next_state_env, dtype=np.float32).flatten()
            next_state = np.reshape(next_state, (1, -1))

            #add to experience memory
            agent.remember(state, action, reward, next_state, done)
            state = next_state
            if done:
                break

        #
        # INSERT CODE HERE to train network if in the memory is more samples then size of the batch
        #
        if len(agent.memory) > batch_size:
            agent.replay(batch_size)
        
        summary.append(total_reward)
    print("epoch #{}\tmean reward = {:.3f}\tepsilon = {:.3f}".format(e, np.mean(summary), agent.epsilon))
    if np.mean(summary) > 0.9:
        print ("You Win!")
        break
    agent.update_epsilon_value()
    

epoch #0	mean reward = 0.000	epsilon = 0.750
epoch #1	mean reward = 0.060	epsilon = 0.675
epoch #2	mean reward = 0.220	epsilon = 0.608
epoch #3	mean reward = 0.130	epsilon = 0.547
epoch #4	mean reward = 0.250	epsilon = 0.492
epoch #5	mean reward = 0.290	epsilon = 0.443
epoch #6	mean reward = 0.380	epsilon = 0.399
epoch #7	mean reward = 0.220	epsilon = 0.359
epoch #8	mean reward = 0.380	epsilon = 0.323
epoch #9	mean reward = 0.490	epsilon = 0.291
epoch #10	mean reward = 0.430	epsilon = 0.262
epoch #11	mean reward = 0.630	epsilon = 0.235
epoch #12	mean reward = 0.740	epsilon = 0.212
epoch #13	mean reward = 0.820	epsilon = 0.191
epoch #14	mean reward = 0.760	epsilon = 0.172
epoch #15	mean reward = 0.850	epsilon = 0.154
epoch #16	mean reward = 0.850	epsilon = 0.139
epoch #17	mean reward = 0.860	epsilon = 0.125
epoch #18	mean reward = 0.890	epsilon = 0.113
epoch #19	mean reward = 0.860	epsilon = 0.101
epoch #20	mean reward = 0.900	epsilon = 0.091
epoch #21	mean reward = 0.840	epsilon = 0.08

Czas przygotować model sieci, która będzie się uczyła działania w środowisku [*CartPool*](https://gym.openai.com/envs/CartPole-v0/):

In [9]:
env = gym.make("CartPole-v0").env
state_size = env.observation_space.shape[0]
action_size = env.action_space.n
learning_rate = 0.001

model = DQNModel(state_size, action_size, learning_rate)

d:\.Astudia\.venv\Lib\site-packages\gym\envs\registration.py:555: UserWarning: WARN: The environment CartPole-v0 is out of date. You should consider upgrading to version `v1`.
  logger.warn(


Czas nauczyć agenta gry w środowisku *CartPool*:
* 1 pkt < 10 epok,
* 0.5 pkt < 20 epok,
* 0.25 pkt - w pozostałych przypadkach.

In [10]:
agent = DQNAgent(action_size, learning_rate, model)

agent.epsilon = 0.75
agent.epsilon_decay = 0.9

done = False
batch_size = 128
EPISODES = 1000
counter = 0
for e in range(EPISODES):
    summary = []
    for _ in range(100):
        total_reward = 0
        env_state = env.reset()
        if isinstance(env_state, tuple):
            env_state = env_state[0]
    
        #
        # INSERT CODE HERE to prepare appropriate format of the state for network
        #
        state = np.reshape(env_state, (1, state_size)).astype(np.float32)
        
        for time in range(300):
            action = agent.get_action(state)
            step_result = env.step(action)
            if len(step_result) == 5:
                next_state_env, reward, terminated, truncated, _ = step_result
                done = terminated or truncated
            else:
                next_state_env, reward, done, _ = step_result
            total_reward += reward

            #
            # INSERT CODE HERE to prepare appropriate format of the next state for network
            #
            next_state = np.reshape(next_state_env, (1, state_size)).astype(np.float32)

            #add to experience memory
            agent.remember(state, action, reward, next_state, done)
            state = next_state
            if done:
                break

        #
        # INSERT CODE HERE to train network if in the memory is more samples then size of the batch
        #
        if len(agent.memory) > batch_size:
            agent.replay(batch_size)
        
        summary.append(total_reward)
    print("epoch #{}\tmean reward = {:.3f}\tepsilon = {:.3f}".format(e, np.mean(summary), agent.epsilon))
    if np.mean(summary) > 195:
        print ("You Win!")
        break
    agent.update_epsilon_value()
    

epoch #0	mean reward = 20.010	epsilon = 0.750
epoch #1	mean reward = 17.540	epsilon = 0.675
epoch #2	mean reward = 21.170	epsilon = 0.608
epoch #3	mean reward = 42.400	epsilon = 0.547
epoch #4	mean reward = 54.590	epsilon = 0.492
epoch #5	mean reward = 81.370	epsilon = 0.443
epoch #6	mean reward = 111.640	epsilon = 0.399
epoch #7	mean reward = 188.040	epsilon = 0.359
epoch #8	mean reward = 185.590	epsilon = 0.323
epoch #9	mean reward = 216.440	epsilon = 0.291
You Win!
